In [ ]:
%pip install -q antropy


In [ ]:
# Dependencies are installed in the previous cell.


In [ ]:
import gc
import os
import re
import shutil
import subprocess
import traceback
from pathlib import Path

import antropy as ant
import numpy as np
import pandas as pd
from mne.filter import filter_data
from numba import njit, prange
from scipy.signal import hilbert, welch
from scipy.stats import entropy, kurtosis, skew


In [ ]:
# No Hugging Face login is required.
# Restore one historical segmentation version under /kaggle/working before running.


In [ ]:
# ==========================================================================
# 0. CONSTANTS, OUTPUTS AND BATCH CONTROL
# ==========================================================================
FS = 256
EEG_INTEGER_SCALE_VOLTS = 1e-7  # 0.1 microvolt/count from segmentation

BANDS = {
    'delta': [0.5, 4.0],
    'theta': [4.0, 8.0],
    'alpha': [8.0, 13.0],
    'beta':  [13.0, 30.0],
    'gamma': [30.0, 100.0],
}

# Historical segmentation versions are restored one at a time with the Kaggle CLI.
# Change SEGMENTATION_VERSION after saving the feature output from the current version.
SEGMENTATION_DATASET_ID = 'cucbuithi/chbmit-segmentation-batch-00'
SEGMENTATION_VERSION = 1
SEGMENTATION_ROOT = f'/kaggle/working/segmentation_v{SEGMENTATION_VERSION}'
AUTO_DOWNLOAD_SEGMENTATION = True

if AUTO_DOWNLOAD_SEGMENTATION and not Path(SEGMENTATION_ROOT).exists():
    print(f'Downloading segmentation version {SEGMENTATION_VERSION}...')
    Path(SEGMENTATION_ROOT).mkdir(parents=True, exist_ok=True)
    subprocess.run([
        'kaggle', 'datasets', 'download',
        f'{SEGMENTATION_DATASET_ID}/{SEGMENTATION_VERSION}',
        '-p', SEGMENTATION_ROOT, '--unzip',
    ], check=True)
    print(f'Restored to {SEGMENTATION_ROOT}')

FEATURE_OUT = '/kaggle/working/Handcrafted'
CONN_OUT_TESTING = '/kaggle/working/CONNECTIVITY/Testing'
CONN_OUT_TRAINING = '/kaggle/working/CONNECTIVITY/Training'
PREICTAL_LABEL = 'preictal'

# Process one included subject per Kaggle version by default. Increase to 2 only
# after confirming runtime and output size for the first subject.
FEATURE_PATIENTS_PER_BATCH = 1
FEATURE_BATCH_NUMBER = 0
SUBJECTS_TO_RUN = None       # Example override: ['chb01']
RESET_FEATURE_OUTPUT = True
MIN_FREE_DISK_GIB = 1.0


In [ ]:
# ==========================================================================
# 1. CONNECTIVITY ESTIMATION FUNCTIONS (corrected PLI and standard wPLI)
# ==========================================================================
def get_pli_matrix(analytic_signal):
    """PLI = |mean(sign(sin(delta phase)))| for each epoch and channel pair."""
    n_ep, n_ch, _ = analytic_signal.shape
    phase = np.angle(analytic_signal)
    out = np.zeros((n_ep, n_ch, n_ch), dtype=np.float64)
    for i in range(n_ch):
        for j in range(i + 1, n_ch):
            signed_lag = np.sign(np.sin(phase[:, i] - phase[:, j]))
            value = np.abs(np.mean(signed_lag, axis=-1))
            out[:, i, j] = out[:, j, i] = value
    return out


def get_wpli_matrix(analytic_signal, eps=1e-12):
    """Standard wPLI using the imaginary analytic cross-spectrum.

    wPLI = |E[Im(X_i conj(X_j))]| / E[|Im(X_i conj(X_j))|].
    This retains amplitude weighting; a phase-only sine ratio is not called wPLI.
    """
    n_ep, n_ch, _ = analytic_signal.shape
    out = np.zeros((n_ep, n_ch, n_ch), dtype=np.float64)
    for i in range(n_ch):
        for j in range(i + 1, n_ch):
            imag_cross = np.imag(analytic_signal[:, i] * np.conj(analytic_signal[:, j]))
            num = np.abs(np.mean(imag_cross, axis=-1))
            den = np.mean(np.abs(imag_cross), axis=-1)
            value = np.divide(num, den, out=np.zeros_like(num), where=den > eps)
            out[:, i, j] = out[:, j, i] = value
    return out


CONNECTIVITY_FUNCS = {'PLI': get_pli_matrix, 'wPLI': get_wpli_matrix}


In [ ]:
# ==========================================================================
# 2. TRADITIONAL FEATURE FUNCTIONS (vectorised over epochs AND channels)
# ==========================================================================
# Every function takes x of shape (n_epochs, n_channels, n_samples)
# and returns an array of shape (n_epochs, n_channels).

FEATURE_NAMES = [
    'mean', 'std', 'skewness', 'kurtosis',
    'mean_spectral_power', 'max_spectral_power', 'freq_of_max_power',
    'zero_crossing_rate', 'shannon_entropy', 'sample_entropy',
]

def shannon_entropy_batch(x, bins=10):
    """Same result as entropy(np.histogram(sig, bins=10, density=True)[0] + 1e-12) for every signal.

    The histogram is rebuilt with the same binning rules as np.histogram
    (uniform bins over [min, max], right-most bin closed), but for all signals at once.
    """
    n_ep, n_ch, n_t = x.shape
    lo, hi = x.min(axis=-1), x.max(axis=-1)
    flat = lo == hi                                  # np.histogram widens a degenerate range by +-0.5
    lo, hi = np.where(flat, lo - 0.5, lo), np.where(flat, hi + 0.5, hi)

    edges = lo[..., None] + np.arange(bins + 1) * ((hi - lo) / bins)[..., None]
    edges[..., -1] = hi

    idx = ((x - lo[..., None]) * (bins / (hi - lo))[..., None]).astype(np.intp)
    idx[idx == bins] -= 1
    idx[x < np.take_along_axis(edges, idx, axis=-1)] -= 1                                  # float-error fix
    idx[(x >= np.take_along_axis(edges, idx + 1, axis=-1)) & (idx != bins - 1)] += 1       # float-error fix

    offset = (np.arange(n_ep * n_ch) * bins).reshape(n_ep, n_ch, 1)
    counts = np.bincount((idx + offset).ravel(), minlength=n_ep * n_ch * bins).reshape(n_ep, n_ch, bins)

    density = counts / np.diff(edges, axis=-1) / counts.sum(axis=-1, keepdims=True)
    return entropy(density + 1e-12, axis=-1)

def sample_entropy_batch(x, order=2):
    n_ep, n_ch, _ = x.shape
    x64 = x.astype(np.float64)
    r = 0.2 * np.std(x64, axis=-1).reshape(-1)
    rows = np.ascontiguousarray(x64.reshape(n_ep * n_ch, -1))
    return _sample_entropy_rows(rows, order, r).reshape(n_ep, n_ch)

@njit
def _sample_entropy_1d(seq, order, r):
    """Identical to antropy._numba_sampen (Chebyshev metric), made callable from prange."""
    size = seq.size
    numerator = 0
    denominator = 0
    for offset in range(1, size - order):
        n_numerator = int(abs(seq[order] - seq[order + offset]) >= r)
        n_denominator = 0
        for idx in range(order):
            d = int(abs(seq[idx] - seq[idx + offset]) >= r)
            n_numerator += d
            n_denominator += d
        if n_numerator == 0:
            numerator += 1
        if n_denominator == 0:
            denominator += 1
        prev_in_diff = int(abs(seq[order] - seq[offset + order]) >= r)
        for idx in range(1, size - offset - order):
            out_diff = int(abs(seq[idx - 1] - seq[idx + offset - 1]) >= r)
            in_diff = int(abs(seq[idx + order] - seq[idx + offset + order]) >= r)
            n_numerator += in_diff - out_diff
            n_denominator += prev_in_diff - out_diff
            prev_in_diff = in_diff
            if n_numerator == 0:
                numerator += 1
            if n_denominator == 0:
                denominator += 1
    if denominator == 0:
        return np.nan
    elif numerator == 0:
        return np.inf
    else:
        return -np.log(numerator / denominator)


@njit(parallel=True)
def _sample_entropy_rows(X, order, r):
    out = np.empty(X.shape[0])
    for k in prange(X.shape[0]):          # every (epoch, channel) signal is independent -> multi-threaded
        out[k] = _sample_entropy_1d(X[k], order, r[k])
    return out

def extract_features_batch(x, fs=FS):
    """Traditional features for ALL epochs and channels at once.

    x: (n_epochs, n_channels, n_samples)
    Returns (n_epochs, n_channels * n_features), columns ordered channel-major:
    [mean_0, std_0, ..., sample_entropy_0, mean_1, ...] (see feature_columns).
    """
    n_t = x.shape[-1]
    freqs, psd = welch(x, fs=fs, nperseg=min(n_t, 256), axis=-1)       # psd: (n_ep, n_ch, n_freqs)

    feats = {
        'mean': x.mean(axis=-1),
        'std': x.std(axis=-1),
        'skewness': skew(x, axis=-1),
        'kurtosis': kurtosis(x, axis=-1),
        'mean_spectral_power': psd.mean(axis=-1),
        'max_spectral_power': psd.max(axis=-1),
        'freq_of_max_power': freqs[psd.argmax(axis=-1)],
        'zero_crossing_rate': np.sum(np.diff(np.sign(x), axis=-1) != 0, axis=-1) / n_t,
        'shannon_entropy': shannon_entropy_batch(x),
        'sample_entropy': sample_entropy_batch(x),
    }
    stacked = np.stack([feats[name] for name in FEATURE_NAMES], axis=-1)   # (n_ep, n_ch, n_feat)
    return stacked.reshape(x.shape[0], -1)


def feature_columns(n_channels):
    return [f'{name}_{ch}' for ch in range(n_channels) for name in FEATURE_NAMES]

In [ ]:
# ==========================================================================
# 3. PIPELINE HELPERS
# ==========================================================================
def natural_key(name):
    return [(0, int(t)) if t.isdigit() else (1, t) for t in re.findall(r'\d+|\D+', name)]

def list_npy(path):
    return sorted((f for f in os.listdir(path) if f.endswith('.npy')), key=natural_key)

def filter_band(epochs, low, high):
    """Band-pass FIR filter along the last axis. epochs: (n_epochs, n_channels, n_samples)."""
    return filter_data(epochs, sfreq=FS, l_freq=low, h_freq=high,
                       method='fir', fir_design='firwin', verbose=False)


def save_connectivity(analytic_epochs, conn_root, rel_dir, band):
    """Compute (all epochs at once) and save every connectivity measure of CONNECTIVITY_FUNCS for one band."""
    for name, get_matrix in CONNECTIVITY_FUNCS.items():
        matrices = get_matrix(analytic_epochs)          # (n_epochs, n_channels, n_channels)
        out_dir = os.path.join(conn_root, f'{name}_matrix', rel_dir)
        os.makedirs(out_dir, exist_ok=True)
        out_path = os.path.join(out_dir, f'{band}.npy')
        free_gib = shutil.disk_usage(out_dir).free / 2**30
        if free_gib < MIN_FREE_DISK_GIB:
            raise OSError(f'Only {free_gib:.2f} GiB free before writing {out_path}')
        np.save(out_path, matrices.astype(np.float32), allow_pickle=False)
        del matrices


def extract_band_features(filtered_epochs, band, seizure_id, epoch_offset=0, label=None):
    """Feature table (one row per epoch) for one band, computed on all epochs at once.

    Column order: features..., Band, SeizureID, Epoch, [Class]
    """
    n_epochs, n_ch, _ = filtered_epochs.shape
    df = pd.DataFrame(extract_features_batch(filtered_epochs), columns=feature_columns(n_ch))
    df['Band'] = band
    df['SeizureID'] = seizure_id
    df['Epoch'] = epoch_offset + np.arange(n_epochs)
    if label is not None:
        df['Class'] = label
    return df


def process_npy_file(npy_path, conn_root, rel_dir, seizure_id, epoch_offset=0, label=None, metadata=None):
    """Filter each band once, then compute connectivity and traditional features from it.

    Returns (feature_df, n_epochs).
    """
    stored = np.load(npy_path)
    # Storage-efficient segmentation uses int16 at 0.1 microvolt/count.
    # Restore SI volts before filtering and feature extraction.
    if np.issubdtype(stored.dtype, np.integer):
        epochs = stored.astype(np.float64) * EEG_INTEGER_SCALE_VOLTS
    else:
        epochs = stored.astype(np.float64)
    del stored
    dfs = []
    for band, (low, high) in BANDS.items():
        filtered = filter_band(epochs, low, high)
        analytic = hilbert(filtered, axis=-1)
        save_connectivity(analytic, conn_root, rel_dir, band)
        dfs.append(extract_band_features(filtered, band, seizure_id, epoch_offset, label))
        del filtered, analytic
    n_epochs = epochs.shape[0]
    out = pd.concat(dfs, ignore_index=True)
    if metadata is not None:
        if len(metadata) != n_epochs:
            raise ValueError(f'{npy_path}: metadata rows {len(metadata)} != epochs {n_epochs}')
        meta = metadata.reset_index(drop=True).copy()
        global_epoch = meta.pop('Epoch').to_numpy()
        meta = meta.drop(columns=['Subject', 'SeizureID'], errors='ignore')
        meta['_row'] = np.arange(n_epochs)
        out = out.merge(meta, left_on='Epoch', right_on='_row', how='left').drop(columns='_row')
        out['Epoch'] = np.tile(global_epoch, len(BANDS))
    del epochs
    gc.collect()
    return out, n_epochs


def save_features(dfs, subject_id, file_name):
    if not dfs:
        print(f'[WARN] no data -> {subject_id}/{file_name} NOT saved')
        return
    out_dir = os.path.join(FEATURE_OUT, subject_id)
    os.makedirs(out_dir, exist_ok=True)
    pd.concat(dfs, ignore_index=True).to_csv(os.path.join(out_dir, file_name), index=False)

In [ ]:
# ==========================================================================
# 3.1 DISCOVER SUBJECTS IN THE RESTORED SEGMENTATION VERSION
# ==========================================================================
def discover_segmented_subjects(input_root):
    """Return {subject: subject_dir} for included, fully saved subjects.

    A testing_epoch_manifest.csv exists only for cohort-included patients.
    Duplicate subjects across attached datasets are rejected to prevent leakage.
    """
    found = {}
    for manifest in Path(input_root).rglob('testing_epoch_manifest.csv'):
        subject_dir = manifest.parent
        subject = subject_dir.name.lower()
        if not re.fullmatch(r'chb\d{2}', subject):
            continue
        required = [
            subject_dir / 'Training' / 'preictal',
            subject_dir / 'Training' / 'interictal',
            subject_dir / 'testing',
            subject_dir / 'segmentation_qc.csv',
        ]
        if not all(p.exists() for p in required):
            print(f'[SKIP INCOMPLETE] {subject_dir}')
            continue
        if subject in found and found[subject] != subject_dir:
            raise RuntimeError(
                f'Duplicate segmented subject {subject}: {found[subject]} and {subject_dir}')
        found[subject] = subject_dir
    if not found:
        raise FileNotFoundError(
            f'No included segmentation subjects found under {input_root}. '
            'Download and unzip the selected historical dataset version first.')
    return dict(sorted(found.items()))


def segmentation_inventory(input_root):
    rows = []
    for path in Path(input_root).rglob('cohort_manifest.csv'):
        df = pd.read_csv(path)
        df['SourceManifest'] = str(path)
        rows.append(df)
    if not rows:
        return pd.DataFrame()
    inventory = pd.concat(rows, ignore_index=True)
    if inventory['Subject'].duplicated().any():
        duplicates = inventory.loc[inventory['Subject'].duplicated(False), 'Subject'].tolist()
        raise RuntimeError(f'Duplicate subjects across cohort manifests: {duplicates}')
    return inventory.sort_values('Subject').reset_index(drop=True)


In [ ]:
# ==========================================================================
# 4. SUBJECT-LEVEL PROCESSING
# ==========================================================================
def process_testing_subject(folder):
    """Process every testing epoch and preserve the phase/mask metadata."""
    subject_id = os.path.basename(os.path.normpath(folder))
    manifest_path = os.path.join(folder, 'testing_epoch_manifest.csv')
    if not os.path.exists(manifest_path):
        raise FileNotFoundError(f'{manifest_path} is required; rerun segmentation corrected v2')
    manifest = pd.read_csv(manifest_path)
    manifest['NpyRelativePath'] = manifest['NpyRelativePath'].str.replace('\\\\', '/', regex=False)

    testing_root = os.path.join(folder, 'testing') if os.path.isdir(os.path.join(folder, 'testing')) else folder
    for seizure_id, smeta in manifest.groupby('SeizureID', sort=True):
        dfs = []
        for rel, meta in smeta.groupby('NpyRelativePath', sort=False):
            npy_path = os.path.join(testing_root, *str(rel).split('/'))
            if not os.path.exists(npy_path):
                raise FileNotFoundError(npy_path)
            parts = Path(rel).parts
            seizure_dir, block_dir, stem = parts[-3], parts[-2], Path(parts[-1]).stem
            rel_dir = os.path.join(subject_id, seizure_dir, block_dir, stem)
            meta = meta.sort_values('EpochInFile')
            df, _ = process_npy_file(
                npy_path, CONN_OUT_TESTING, rel_dir,
                seizure_id=seizure_id, epoch_offset=0, metadata=meta)
            dfs.append(df)
        combined = pd.concat(dfs, ignore_index=True)
        for block, block_df in combined.groupby('Block', sort=True):
            save_features([block_df], subject_id,
                          f'seizure_{int(seizure_id):02d}_block_{int(block):02d}.csv')
        del dfs
        gc.collect()


def process_training_subject(folder):
    folder = Path(folder)
    subject_id = folder.parent.name if folder.name == 'Training' else folder.name
    print(f'Processing {subject_id} (training)')

    for label in sorted(d for d in os.listdir(folder) if os.path.isdir(os.path.join(folder, d))):
        label_path = os.path.join(folder, label)
        
        if label == PREICTAL_LABEL:
            dfs = []
            for fname in list_npy(label_path):
                stem = os.path.splitext(fname)[0]
                df, _ = process_npy_file(
                    os.path.join(label_path, fname), CONN_OUT_TRAINING,
                    os.path.join(subject_id, label, stem), seizure_id=stem, label=label)
                dfs.append(df)

            save_features(dfs, subject_id, f'{label}.csv')
            del dfs
        else:
            for sz in sorted(d for d in os.listdir(label_path) if os.path.isdir(os.path.join(label_path, d))):
                sz_path = os.path.join(label_path, sz)
                epoch_offset = 0
                dfs = []
                for fname in list_npy(sz_path):
                    stem = os.path.splitext(fname)[0]
                    df, n_epochs = process_npy_file(
                        os.path.join(sz_path, fname), CONN_OUT_TRAINING,
                        os.path.join(subject_id, label, sz, stem),
                        seizure_id=sz, epoch_offset=epoch_offset, label=label)
                    dfs.append(df)
                    epoch_offset += n_epochs
                save_features(dfs, subject_id, f'{sz}_{label}.csv')
                del dfs
        gc.collect()

In [ ]:
# ==========================================================================
# 5. RUN ONE FEATURE-EXTRACTION BATCH
# ==========================================================================
if not Path(SEGMENTATION_ROOT).exists():
    raise FileNotFoundError(
        f'{SEGMENTATION_ROOT} does not exist. Download and unzip dataset version '
        f'{SEGMENTATION_VERSION} before running feature extraction.')

subject_dirs = discover_segmented_subjects(SEGMENTATION_ROOT)
inventory = segmentation_inventory(SEGMENTATION_ROOT)
print(f'Segmentation source: version {SEGMENTATION_VERSION} -> {SEGMENTATION_ROOT}')
print('Segmentation cohort inventory:')
display(inventory)
print('Included subjects discovered:', list(subject_dirs))

if SUBJECTS_TO_RUN is not None:
    selected = [s.lower() for s in SUBJECTS_TO_RUN]
    missing = [s for s in selected if s not in subject_dirs]
    if missing:
        raise ValueError(f'Subjects not found among included inputs: {missing}')
else:
    all_subjects = list(subject_dirs)
    lo = FEATURE_BATCH_NUMBER * FEATURE_PATIENTS_PER_BATCH
    selected = all_subjects[lo:lo + FEATURE_PATIENTS_PER_BATCH]

if not selected:
    raise ValueError('This feature batch selects no subject')
print(f'Feature batch {FEATURE_BATCH_NUMBER}: {selected}')

if RESET_FEATURE_OUTPUT:
    for out_dir in [FEATURE_OUT, '/kaggle/working/CONNECTIVITY']:
        if os.path.isdir(out_dir):
            print('[RESET OUTPUT]', out_dir)
            shutil.rmtree(out_dir)

run_rows = []
for subject in selected:
    folder = subject_dirs[subject]
    try:
        print(f'\n===== {subject}: TESTING =====')
        process_testing_subject(str(folder))
        print(f'\n===== {subject}: TRAINING =====')
        process_training_subject(str(folder / 'Training'))
        run_rows.append({'Subject': subject, 'Status': 'completed', 'Error': ''})
    except Exception as exc:
        traceback.print_exc()
        run_rows.append({'Subject': subject, 'Status': 'failed', 'Error': repr(exc)})

run_manifest = pd.DataFrame(run_rows)
run_manifest.to_csv('/kaggle/working/feature_extraction_manifest.csv', index=False)
display(run_manifest)

if (run_manifest['Status'] != 'completed').any():
    raise RuntimeError('At least one subject failed; inspect the traceback before saving output')
print('Feature extraction batch completed. Save this output before running the next batch.')
